# ِDataset MetaData

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats


import random

# Read Dataset

In [ ]:

df=pd.read_csv('/kaggle/input/marketing-ab-testing/marketing_AB.csv')
df.head()

# Explore Dataset

In [ ]:
df.shape

In [ ]:
df.head(5)

In [ ]:
df.tail(5)

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.columns

# Data cleansing

In [ ]:
if "Unnamed: 0" in df.columns:
   
    df = df.drop(["Unnamed: 0"], axis =1)

In [ ]:
df.rename(columns=lambda x: x.strip().replace(" ", "_"), inplace=True)
df.head(1)

In [ ]:
#duplicate rows?
dups = df.duplicated()
print(dups.any())

In [ ]:
#duplicated user_id?
df[df["user_id"].duplicated()].count()

In [ ]:
print(f'Rows            : {df.shape[0]}')
print(f'Columns         : {df.shape[1]}')
print(f'Features        : {df.columns.tolist()}')
print(f'Missing Values  : {df.isnull().values.sum()}')
print(f'Unique Values   : \n\n{df.nunique()}')

In [ ]:
df.isnull().any()

In [ ]:
df["converted_int"] = df['converted'].apply(lambda x:1 if x== True  else  0)
df["converted_int"].sum()

In [ ]:
df.head(5)

# EDA

In [ ]:
def val_count(column):
    plt.figure(figsize=(7,5))
    sns.countplot(data=df, x=column)
    plt.title(f'Value Count - {column}')
    plt.show()

    #print(df[column].value_counts())

In [ ]:
#create columns of interest
col_lst = df.columns[[1,2,4,5]]

#loop through columns of interest
for i in col_lst:
    val_count(i)

the majority of users saw the ads compared to those saw the psa

the most ads seen by a user occurred on Friday and then Monday

between 10AM and 3PM, users saw the most ads

In [ ]:
grouped_counts = df.groupby('test_group')['converted'].value_counts()
grouped_counts

# Plot the data as a pie chart
grouped_counts.plot.pie(figsize= (3,3),autopct='%1.1f%%')

# Set the title and axis labels
plt.title('Distribution of Converted by Test Group')
plt.ylabel('')

# Show the plot
plt.show()


In [ ]:
ax =df.groupby(by =['most_ads_day', 'test_group']).sum()['converted_int'].unstack('test_group').plot(kind= 'bar', figsize= (4,3), grid= True, stacked= True)
ax.set_ylabel('converted_int')
ax.set_title('Most ad days and converted_int')
plt.show()

In [ ]:
#visualize conversion by day
df_day_conv = pd.DataFrame(df.groupby('most_ads_day')['converted'].mean())
df_day_conv.reset_index(inplace=True)

plt.figure(figsize=(5,3))
plt.bar(data=df_day_conv, x='most_ads_day', height='converted')
plt.title('Conversion Rate by Day')
plt.axhline(df['converted'].mean(), color='r', linestyle='--', label='average')
plt.legend()
plt.show()

# AB test

### Minimum Sample Size

In [ ]:
import math

def sample_size_calculator(population_size, confidence_level, margin_of_error):
    z_score = {
        0.90: 1.645,
        0.95: 1.96,
        0.99: 2.576
    }
    
    z = z_score[confidence_level]
    p = 0.5 # assuming 50% for a conservative estimate of the sample size
    q = 1 - p
    
    sample_size = ((z**2) * p * q * population_size) / ((z**2 * q) + ((margin_of_error**2) * (population_size - 1)))
    return math.ceil(sample_size)

In [ ]:
#compute sample size 
pop_size = df.shape[0]
conf_level = 0.99
margin_err = 0.03

sample_size = sample_size_calculator(pop_size, conf_level, margin_err)
print(f"The population size is :{pop_size} \nThe required sample size is: {sample_size}")

In [ ]:
treatment = df.query('test_group == "ad"')
control = df.query('test_group == "psa"')


In [ ]:
df.converted.mean() *100

In [ ]:
control["converted"].mean()


In [ ]:
treatment["converted"].mean()

you can see the the mean of converted in treatment group is more than control group 
which make the hypothsis of ad has positive effect on conversion 

In [ ]:
ad_converted = np.random.binomial(len(treatment),
                                  df.converted.mean(), 10000) / len(treatment)

plt.hist(ad_converted, bins=50)

In [ ]:
psa_converted = np.random.binomial(len(control),
                                   df.converted.mean(), 10000) / len(control)

plt.hist(psa_converted, bins=50 ,align = 'mid' , data=None)

In [ ]:
p_diffs = ad_converted - psa_converted
p_diffs
p_diffs.mean()

In [ ]:
#real diff
# customer who is shown ads bought less
ab_data_diff = treatment['converted'].mean() - control['converted'].mean()
ab_data_diff


Does showing ads to people lead to more purchases? Is this statistically significant or not?
Null Hypothesis (H0): Showing ads has no significant effect on the number of purchases.
Alternative Hypothesis (H1): Showing ads has a significant effect on the number of purchases.

In [ ]:
# is buying less is 
if ab_data_diff > 0 : 
    p_value = (p_diffs > ab_data_diff).mean() * 100
else :
    p_value = (p_diffs <= ab_data_diff).mean() * 100
print ( "p_value = " ,p_value)
if p_value < 0.05:
    print( 'Reject the null hypothesis. There is a significant difference between the two groups.')
else:
    print( 'Fail to reject the null hypothesis. There is no significant difference between the two groups.')


In [ ]:
plt.hist(p_diffs, bins=100)
low = ab_data_diff
higth = p_diffs.mean()
plt.axvline(x=low, color='g')
plt.axvline(x=higth, color='r')

# AB test using Code Simulation

In [ ]:
treatment = df.query('test_group == "ad"')
control = df.query('test_group == "psa"')


In [ ]:
dif = treatment["converted"].mean() - control["converted"].mean()
dif

In [ ]:
conversion = np.array([
    np.append(np.zeros(len(control["converted"])), np.ones(len(treatment["converted"]))),
    np.append(control["converted"],treatment["converted"])
])
conversion_t = conversion.T
conversion_t

In [ ]:

def sh_exp(N):
    experiment_diff_mean = np.empty([N, 1])
    for times in np.arange(N):
        experiment_label = np.random.randint(0, 2, len(conversion_t))
        experiment_data = np.array([
            experiment_label,
            conversion_t[:, 1]
        ]).T
        experiment_diff_mean[times] = experiment_data[experiment_data[:, 0] == 1][:, 1].mean() - experiment_data[experiment_data[:, 0] == 0][:, 1].mean()
    return experiment_diff_mean


In [ ]:
n=2000
p_difs = sh_exp(n)

if dif > 0  : 
    p_value = len(p_difs[p_difs >= dif]) / n * 100
else : 
    p_value = len(p_difs[p_difs <= dif]) / n * 100
    
if p_value < 0.05:
    print( 'Reject the null hypothesis. There is a significant difference between the two groups.')
else:
    print( 'Fail to reject the null hypothesis. There is no significant difference between the two groups.')


    

In [ ]:
sns.displot(p_difs, bins=50)

# T-test 

#### check distribution of two groups by KDE

In [ ]:



control_mean = df.loc[df['test_group'] == 'psa', 'converted'].mean()
treatment_mean = df.loc[df['test_group'] == 'ad', 'converted'].mean()

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x='converted', hue='test_group', fill=True, common_norm=False)
plt.axvline(control_mean, color='b', linestyle='--', label='Control Mean')
plt.axvline(treatment_mean, color='r', linestyle='--', label='Treatment Mean')
plt.title('Distribution of Mean Converted Column by Test Group')
plt.xlabel('Converted')
plt.ylabel('Density')
plt.legend(title='Test Group')
plt.show()


##### check distribution of each group using bootstrap

In [ ]:
boot_treatment=[]
for i in range(1000):
    boot_mean = treatment.sample(frac=1, replace=True)['converted'].mean()
    boot_treatment.append(boot_mean)
boot_treatment=pd.DataFrame(boot_treatment)
boot_treatment.plot(kind='density');

boot_control=[]

for i in range(1000):
    boot_mean=control.sample(frac=1,replace=True)['converted'].mean()
    boot_control.append(boot_mean)
    
boot_control=pd.DataFrame(boot_control)
boot_control.plot(kind='density');
    

In [ ]:

def t_test(treatment_data, control_data) :
    
    t_statistic, p_value = stats.ttest_ind(treatment_data, control_data)
    print("T-statistic:", t_statistic)
    print("P-value:", p_value)
    
    if p_value < 0.05 :    
        print( 'Reject the null hypothesis. There is a significant difference between the two groups.')
    else:
        print( 'Fail to reject the null hypothesis. There is no significant difference between the two groups.')

    return t_statistic, p_value
    

# Perform a t-test to compare the means of the two groups
t_statistic, p_value = t_test(treatment["converted"], control["converted"])

    

# Chi2 Test

Does the display of advertisements correlate with the number of purchases or not?

In [ ]:
ct = pd.crosstab(df['test_group'], df['converted'], margins=True)


ct

In [ ]:

d = np.array([ct.iloc[0][: -1].values, ct.iloc[1][: -1].values])
d

In [ ]:
chi2, p_value, dof, expected  =stats.chi2_contingency(ct)

# Print the results
print("Chi-square statistic:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", dof)
print("Expected frequencies:")
print(expected)

In [ ]:
if p_value < 0.05:
    print ( 'Reject the null hypothesis. There is a significant difference between the two groups.')
else:
    print ('Fail to reject the null hypothesis. There is no significant difference between the two groups.')

  

In [ ]:
# Simin Alavizadeh